# CGC + Chiller Test

This test is for testing the water supply from the chiller to all water cooled cgc devices.

## Import, Setup Logger, Create Instances

In [1]:
# Import required modules
import sys
import os
import logging
from datetime import datetime
from pathlib import Path

# Add src to path
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

from devices.cgc.psu.psu import PSU
from devices.cgc.sw.sw import SW
from devices.cgc.sw_HR.sw_HR import SWHR
from devices.chiller.chiller import Chiller, ChillerCommands

In [2]:
# Setup external logger
repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"016_Temp_Check_GC_Chiller_{timestamp}.log"

logger = logging.getLogger(f"016_Temp_Check_GC_Chiller_{timestamp}")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(console_handler)

logger.info("Logger initialized.")
print(f"Log file: {log_file}")

2026-03-20 13:06:19,086 - INFO - Logger initialized.


Log file: C:\Users\ESIBDlab\PycharmProjects\esibd_bs\debugging\logs\016_Temp_Check_GC_Chiller_20260320_130619.log


## Create Device Instances

In [ ]:
psu1 = PSU("psu1", com=0, port=0, logger=logger)
psu2 = PSU("psu2", com=0, port=0, logger=logger)
psu3 = PSU("psu3", com=0, port=0, logger=logger)
psu4 = PSU("psu4", com=0, port=0, logger=logger)

In [ ]:
swA = SWHR("swA", com=0, stream=0, logger=logger)

In [ ]:
swB = SW("swB", com=0, port=0, logger=logger)

In [ ]:
chiller = Chiller("cgc_chiller", port="COM0", baudrate=9600, logger=logger)

## Connect All Devices

In [ ]:
psu1.connect()
psu2.connect()
psu3.connect()
psu4.connect()

In [ ]:
swA.connect()
swB.connect()

In [ ]:
chiller.connect()

## Configure and Start Chiller

In [ ]:
chiller.set_temperature(9.0)

In [ ]:
chiller.start_device()

## Start Housekeeping

In [ ]:
psu1.start_housekeeping(1)
psu2.start_housekeeping(1)
psu3.start_housekeeping(1)
psu4.start_housekeeping(1)

In [ ]:
swA.start_housekeeping(1)
swB.start_housekeeping(1)

In [ ]:
chiller.start_housekeeping(1)

## Live Temperature Monitoring

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import clear_output
import time

INTERVAL = 5  # seconds between readings

timestamps = []
chiller_temps = []
psu_temps = {name: {f"Sensor {i}": [] for i in range(3)} for name in ["psu1", "psu2", "psu3", "psu4"]}
sw_temps = {name: {f"Sensor {i}": [] for i in range(3)} for name in ["swA", "swB"]}

psus = {"psu1": psu1, "psu2": psu2, "psu3": psu3, "psu4": psu4}
sws = {"swA": swA, "swB": swB}

while True:
    now = datetime.now()
    timestamps.append(now)

    # Read chiller temperature
    try:
        chiller_temps.append(chiller.read_temp())
    except Exception:
        chiller_temps.append(float("nan"))

    # Read PSU sensor temperatures
    for name, dev in psus.items():
        try:
            with dev.thread_lock:
                status, t0, t1, t2 = dev.get_sensor_data()
            for i, t in enumerate([t0, t1, t2]):
                psu_temps[name][f"Sensor {i}"].append(t)
        except Exception:
            for i in range(3):
                psu_temps[name][f"Sensor {i}"].append(float("nan"))

    # Read SW sensor temperatures
    for name, dev in sws.items():
        try:
            with dev.thread_lock:
                status, t0, t1, t2 = dev.get_sensor_data()
            for i, t in enumerate([t0, t1, t2]):
                sw_temps[name][f"Sensor {i}"].append(t)
        except Exception:
            for i in range(3):
                sw_temps[name][f"Sensor {i}"].append(float("nan"))

    # Plot
    clear_output(wait=True)
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

    # Chiller
    ax1.plot(timestamps, chiller_temps, "b-")
    ax1.set_ylabel("Temperature//degC")
    ax1.set_title("Chiller")
    ax1.grid(True)

    # PSUs
    for name in psus:
        for sensor, vals in psu_temps[name].items():
            ax2.plot(timestamps, vals, label=f"{name} {sensor}")
    ax2.set_ylabel("Temperature//degC")
    ax2.set_title("PSU 1-4")
    ax2.legend(loc="upper right", fontsize=7, ncol=4)
    ax2.grid(True)

    # Switches
    for name in sws:
        for sensor, vals in sw_temps[name].items():
            ax3.plot(timestamps, vals, label=f"{name} {sensor}")
    ax3.set_ylabel("Temperature//degC")
    ax3.set_title("swA (HR) / swB")
    ax3.set_xlabel("Time")
    ax3.legend(loc="upper right", fontsize=7, ncol=3)
    ax3.grid(True)

    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

    time.sleep(INTERVAL)

## Stop Housekeeping

In [ ]:
psu1.stop_housekeeping()
psu2.stop_housekeeping()
psu3.stop_housekeeping()
psu4.stop_housekeeping()
swA.stop_housekeeping()
swB.stop_housekeeping()
chiller.stop_housekeeping()

## Stop Chiller and Disconnect

In [ ]:
chiller.stop_device()

In [ ]:
psu1.disconnect()
psu2.disconnect()
psu3.disconnect()
psu4.disconnect()
swA.disconnect()
swB.disconnect()
chiller.disconnect()